In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from scipy.stats import norm

from utils import delta_call, svi_iv, nn_iv, heston_iv

In [15]:
df = pd.read_csv("clean_iv_surface_input.csv")
df['date'] = pd.to_datetime(df['date'])
df = df[df["option_type"] == "call"]

r = 0.04
q = 0

df['w'] = (df['IV']**2)*df['T']
df["F"] = df["underlying_price"] * np.exp((r - q) * df["T"])
df["k"] = np.log(df["strike"] / df["F"])


start = '2026-05-11'
end = "2026-06-12"


# get stock values of AAPL

import yfinance as yf

stock_prices = yf.download(
    "AAPL",
    start=start,
    end = pd.Timestamp(end)+ pd.Timedelta(days=1),
    auto_adjust=True
)

stock_prices = stock_prices['Close'].copy()

dates = sorted(df["date"].unique())
dates.append(pd.Timestamp(end))

stock_path = [stock_prices.loc[date] for date in dates]
stock_path = np.asarray(stock_path, dtype=float)

# options that we can hedge 

call_option_df = df[
    (df["date"] == start) & 
    (df['expiry'] == end) &
    (df["option_type"] == "call")
    ]

[*********************100%***********************]  1 of 1 completed


In [ ]:
results = []

models = ["bs","svi", "nn", "heston"]   # add more later

for model in models:

    for opt_idx in range(len(call_option_df)):

        option = call_option_df.iloc[opt_idx]

        S = stock_path.ravel()

        sigma_sticky = option["IV"]
        
        K = option["strike"]
        T = option["T"]
        k = option["k"]
        r = 0.04
        C0 = option["market_price"]

        n_steps = len(S) - 1
        dt = T / n_steps
        times = np.linspace(0, T, n_steps + 1)

        if model == "bs":
            sigmas_call = sigma_sticky * np.ones(n_steps)


        elif model == "svi":
            sigmas_call = np.array([
                svi_iv(dates[i], K, S[i], T - times[i])
                for i in range(n_steps)
            ], dtype=float)

        elif model == 'heston':
            sigmas_call = np.array([
                heston_iv(dates[i], S[i], K, T - times[i], r)
                for i in range(n_steps)
            ], dtype=float)


        elif model == "nn":
            sigmas_call = np.array([
                nn_iv(str(dates[i]), k, T - times[i], S[i], is_call = 1)
                for i in range(n_steps)
            ], dtype = float)
        

        deltas_call = delta_call(
            S[:n_steps],
            K,
            (T - times)[:n_steps],
            sigmas_call
        )

        call_payout_discounted = np.maximum(S[-1] - K, 0) * np.exp(-r * T)

        stock_profit = np.sum(
            (S[1:] - S[:-1] * np.exp(r * dt))
            * np.exp(-r * times[1:])
            * deltas_call
        )

        hedging_error = call_payout_discounted - stock_profit - C0
        relative_error = abs(hedging_error) / C0

        results.append({
            "model": model,
            "option_index": opt_idx,
            "strike": K,
            "T": T,
            "market_price": C0,
            "hedging_error": hedging_error,
            "relative_error": relative_error,
        })

results_df = pd.DataFrame(results)

In [17]:
summary_df = (
    results_df
    .groupby("model")
    .agg(
        mean_abs_error=("hedging_error", lambda x: np.mean(np.abs(x))),
        rmse=("hedging_error", lambda x: np.sqrt(np.mean(x**2))),
        mean_relative_error=("relative_error", "mean"),
        std_relative_error=("relative_error", "std"),
        median_relative_error=("relative_error", "median"),
        max_relative_error=("relative_error", "max"),
    )
    .reset_index()
)

summary_df

,model,mean_abs_error,rmse,mean_relative_error,std_relative_error,median_relative_error,max_relative_error
0,bs,0.864425,0.962944,0.155532,0.194148,0.070343,0.683632
1,heston,2.885646,3.201783,0.442618,0.457989,0.322118,1.522330
2,nn,1.478285,1.807880,0.154111,0.092008,0.136822,0.342404
3,svi,1.231180,1.619903,0.113616,0.091132,0.104032,0.331737
